# BRANCH 1 : ETL (Extract, Transform, Load)
**Responsable :** Matthias Defretin

# Preprocessing
Author : Matthias Defretin

## 1. Import des librairies

In [2]:
import pandas as pd
import os

## 2. Extract Datas

In [ ]:
path = 'DATA'

df_app = pd.read_csv(os.path.join(path, 'application_train-LFS.txt'))

dfBB = pd.read_csv(os.path.join(path, 'bureau_balance-LFS.txt'))
dfB = pd.read_csv(os.path.join(path, 'bureau-LFS.txt'))
dfCCB = pd.read_csv(os.path.join(path, 'credit_card_balance-LFS.txt'))
dfPA = pd.read_csv(os.path.join(path, 'previous_application-LFS.txt'))
dfPOS = pd.read_csv(os.path.join(path, 'POS_CASH_balance-LFS.txt'))
dfIP = pd.read_csv(os.path.join(path, 'installments_payments-LFS.txt'))

In [ ]:
dfBB.head()

In [ ]:
dfB.head()

In [ ]:
dfCCB.head()

In [ ]:
dfPA.head()

In [ ]:
dfPOS.head()

In [ ]:
dfIP.head()

## 3. Transform & Merge (ETL)
Aggregation des tables secondaires et fusion avec la table principale.

In [ ]:
import numpy as np
import gc

def aggregate_numeric(df, group_var, df_name):
    numeric_df = df.select_dtypes('number')
    if group_var not in numeric_df.columns:
        numeric_df[group_var] = df[group_var]
    
    agg = numeric_df.groupby(group_var).agg(['count', 'mean', 'max', 'min', 'sum'])
    
    columns = []
    for var in agg.columns.levels[0]:
        if var != group_var:
            for stat in agg.columns.levels[1]:
                columns.append('%s_%s_%s' % (df_name, var, stat))
    
    agg.columns = columns
    return agg

print("Agrégation de bureau_balance...")
bureau_balance_agg = aggregate_numeric(dfBB, 'SK_ID_BUREAU', 'bureau_balance')
del dfBB
gc.collect()

print("Fusion bureau et bureau_balance...")
dfB = dfB.merge(bureau_balance_agg, left_on='SK_ID_BUREAU', right_index=True, how='left')
del bureau_balance_agg
gc.collect()

print("Agrégation de bureau...")
bureau_agg = aggregate_numeric(dfB, 'SK_ID_CURR', 'bureau')
del dfB
gc.collect()

print("Agrégation de previous_application...")
prev_agg = aggregate_numeric(dfPA, 'SK_ID_CURR', 'prev')
del dfPA
gc.collect()

print("Agrégation de POS_CASH_balance...")
pos_agg = aggregate_numeric(dfPOS, 'SK_ID_CURR', 'pos')
del dfPOS
gc.collect()

print("Agrégation de installments_payments...")
ins_agg = aggregate_numeric(dfIP, 'SK_ID_CURR', 'installments')
del dfIP
gc.collect()

print("Agrégation de credit_card_balance...")
cc_agg = aggregate_numeric(dfCCB, 'SK_ID_CURR', 'cc')
del dfCCB
gc.collect()

print("Fusion finale...")
df_final = df_app.merge(bureau_agg, on='SK_ID_CURR', how='left')
del bureau_agg
gc.collect()

df_final = df_final.merge(prev_agg, on='SK_ID_CURR', how='left')
del prev_agg
gc.collect()

df_final = df_final.merge(pos_agg, on='SK_ID_CURR', how='left')
del pos_agg
gc.collect()

df_final = df_final.merge(ins_agg, on='SK_ID_CURR', how='left')
del ins_agg
gc.collect()

df_final = df_final.merge(cc_agg, on='SK_ID_CURR', how='left')
del cc_agg
gc.collect()

print(f"Taille du dataset final : {df_final.shape}")
df_final.head()

Optimisation df_app...
Mémoire initiale: 286.23 MB
Mémoire après optim: 92.38 MB (Baisse de 67.7%)
Agrégation de bureau_balance...
Mémoire initiale: 37.42 MB
Mémoire après optim: 11.69 MB (Baisse de 68.8%)
Fusion bureau et bureau_balance...
Agrégation de bureau...
Mémoire initiale: 206.48 MB
Mémoire après optim: 69.12 MB (Baisse de 66.5%)
Agrégation de previous_application...
Mémoire initiale: 261.11 MB
Mémoire après optim: 90.81 MB (Baisse de 65.2%)
Agrégation de POS_CASH_balance...
Mémoire initiale: 79.76 MB
Mémoire après optim: 25.09 MB (Baisse de 68.5%)
Agrégation de installments_payments...
Mémoire initiale: 93.27 MB
Mémoire après optim: 34.65 MB (Baisse de 62.8%)
Agrégation de credit_card_balance...
Mémoire initiale: 83.75 MB
Mémoire après optim: 31.50 MB (Baisse de 62.4%)
Fusion finale...
Taille du dataset final : (307511, 482)


C:\Users\EkiaN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
C:\Users\EkiaN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,cc_SK_DPD_count,cc_SK_DPD_mean,cc_SK_DPD_max,cc_SK_DPD_min,cc_SK_DPD_sum,cc_SK_DPD_DEF_count,cc_SK_DPD_DEF_mean,cc_SK_DPD_DEF_max,cc_SK_DPD_DEF_min,cc_SK_DPD_DEF_sum
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,6.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Nettoyage des données
Gestion des duplicatas et valeurs manquantes.

In [ ]:
print(f"Nombre de duplicatas avant : {df_final.duplicated().sum()}")
df_final = df_final.drop_duplicates()

import numpy as np
df_final = df_final.replace([np.inf, -np.inf], np.nan)

for col in df_final.select_dtypes('number').columns:
    if df_final[col].isnull().any():
        df_final[col] = df_final[col].fillna(df_final[col].median())

for col in df_final.select_dtypes('object').columns:
    if df_final[col].isnull().any():
        df_final[col] = df_final[col].fillna(df_final[col].mode()[0])

print("Nettoyage terminé.")
print(f"Valeurs manquantes restantes : {df_final.isnull().sum().sum()}")

Nombre de duplicatas avant : 0
Nettoyage terminé.
Valeurs manquantes restantes : 0


## 5. Traitement des valeurs aberrantes
Correction des anomalies connues (ex: DAYS_EMPLOYED) et traitement statistique.

In [ ]:
print("Traitement de DAYS_EMPLOYED...")

df_final = df_final.copy()

df_final['DAYS_EMPLOYED_ANOM'] = df_final["DAYS_EMPLOYED"] == 365243

df_final['DAYS_EMPLOYED'] = df_final['DAYS_EMPLOYED'].replace({365243: np.nan})

df_final['DAYS_EMPLOYED'] = df_final['DAYS_EMPLOYED'].fillna(df_final['DAYS_EMPLOYED'].median())

print("Anomalies traitées.")
df_final['DAYS_EMPLOYED'].describe()

Traitement de DAYS_EMPLOYED...
Anomalies traitées.


count    307511.000000
mean      -2251.606131
std        2136.193492
min      -17912.000000
25%       -2760.000000
50%       -1648.000000
75%        -933.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

## 6. Feature Engineering
Création de nouvelles variables métiers (ratios financiers).

In [ ]:
new_features = pd.DataFrame(index=df_final.index)

new_features['PAYMENT_RATE'] = df_final['AMT_ANNUITY'] / df_final['AMT_CREDIT']
new_features['ANNUITY_INCOME_PERC'] = df_final['AMT_ANNUITY'] / df_final['AMT_INCOME_TOTAL']
new_features['CREDIT_INCOME_PERC'] = df_final['AMT_CREDIT'] / df_final['AMT_INCOME_TOTAL']
new_features['INCOME_PER_PERSON'] = df_final['AMT_INCOME_TOTAL'] / df_final['CNT_FAM_MEMBERS']
new_features['INCOME_CREDIT_PERC'] = df_final['AMT_INCOME_TOTAL'] / df_final['AMT_CREDIT']

df_final = pd.concat([df_final, new_features], axis=1)

print("Nouvelles variables créées.")
df_final[['PAYMENT_RATE', 'ANNUITY_INCOME_PERC', 'CREDIT_INCOME_PERC', 'INCOME_PER_PERSON']].head()

Nouvelles variables créées.


,PAYMENT_RATE,ANNUITY_INCOME_PERC,CREDIT_INCOME_PERC,INCOME_PER_PERSON
0,0.060749,0.121978,2.007889,202500.0
1,0.027598,0.132217,4.790750,135000.0
2,0.050000,0.100000,2.000000,67500.0
3,0.094941,0.219900,2.316167,67500.0
4,0.042623,0.179963,4.222222,121500.0


## 7. Gestion de la multi-colinéarité
Identification et suppression des variables fortement corrélées (> 0.9).

In [ ]:
numeric_cols = df_final.select_dtypes('number').columns
corr_matrix = df_final[numeric_cols].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

print(f"Nombre de colonnes à supprimer (corrélation > 0.9) : {len(to_drop)}")
print(f"Colonnes supprimées (extrait) : {to_drop[:10]}")

df_final = df_final.drop(columns=to_drop)

print(f"Taille du dataset après suppression des corrélations : {df_final.shape}")

Nombre de colonnes à supprimer (corrélation > 0.9) : 201
Colonnes supprimées (extrait) : ['AMT_GOODS_PRICE', 'REGION_RATING_CLIENT_W_CITY', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE']
Taille du dataset après suppression des corrélations : (307511, 287)


**Note sur la suppression des colonnes :**
La suppression de 201 colonnes est normale pour ce dataset. Elle est due à deux facteurs principaux :
1.  **Redondance structurelle** : Le dataset contient les mêmes informations sur les bâtiments sous 3 formes : Moyenne (`_AVG`), Mode (`_MODE`) et Médiane (`_MEDI`). Ces 3 versions sont corrélées à quasi 100%.
2.  **Agrégations** : Lors de la fusion des tables, nous avons créé des `mean`, `sum`, `max` qui peuvent être très corrélés (ex: si un client n'a qu'un seul crédit, `mean` = `max` = `min`).

Garder ces variables n'apporterait pas d'information supplémentaire et risquerait de perturber le modèle de Régression Logistique (instabilité des coefficients).

## 8. Sauvegarde du dataset nettoyé
Sauvegarde en format CSV (ou Pickle pour conserver les types) pour l'étape suivante.

In [ ]:
output_path = os.path.join(path, 'application_train_cleaned.csv')
print(f"Sauvegarde du fichier nettoyé vers : {output_path}")
df_final.to_csv(output_path, index=False)

print("Sauvegarde terminée.")

Sauvegarde du fichier nettoyé vers : DATA\application_train_cleaned.csv
Sauvegarde terminée.
